In [ ]:
# Cell 1 - Import Libraries & Load Data
print("📥 MEMUAT LIBRARY DAN DATA...")

import pandas as pd
import numpy as np
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

print("✅ Libraries berhasil diimport!")

# Load data
try:
    df = pd.read_csv('comments.csv')
    print(f"✅ Berhasil load data: {len(df)} komentar")
    print(f"📊 Kolom: {list(df.columns)}")
    
    # Tampilkan sample
    print("\n📋 Sample data awal:")
    display(df.head(3))
    
except FileNotFoundError:
    print("❌ ERROR: File 'comments.csv' tidak ditemukan!")
except Exception as e:
    print(f"❌ ERROR: {e}")

In [42]:
# Cell 2 - Hapus Duplikasi
print("🧹 MENGHAPUS DUPLIKASI...")

print(f"📊 Data sebelum: {len(df)} komentar")

# Hapus duplikat
df = df.drop_duplicates()
print(f"✅ Duplikat dihapus")

# Hapus duplikat komentar
df = df.drop_duplicates(subset=['comment'])
print(f"✅ Duplikat komentar dihapus")

print(f"📊 Data sesudah: {len(df)} komentar")

🧹 MENGHAPUS DUPLIKASI...
📊 Data sebelum: 1139 komentar
✅ Duplikat dihapus
✅ Duplikat komentar dihapus
📊 Data sesudah: 1132 komentar


In [ ]:
# Cell Debug - Cek Data Asli
print("🔍 DEBUG: CEK DATA ASLI")

# Cek beberapa komentar pertama yang mengandung karakter khusus
print("1. CEK KOMENTAR DENGAN KARAKTER KHUSUS:")
for i in range(min(10, len(df))):
    comment = df.iloc[i]['comment']
    if any(ord(char) > 127 for char in str(comment)):
        print(f"   Row {i}: {comment}")
        # Tampilkan karakter problematic
        problematic = [(char, ord(char)) for char in str(comment) if ord(char) > 127]
        print(f"   Karakter problematic: {problematic}")

print("\n2. CEK TIPE DATA:")
print(f"   Tipe df: {type(df)}")
print(f"   Tipe kolom comment: {df['comment'].dtype}")

print("\n3. SAMPLE DATA ASLI:")
print(df[['comment']].head(3))

In [ ]:
# Cell Debug - Test Cleaning Step by Step
print("🔍 DEBUG: TEST CLEANING STEP BY STEP")

# Ambil 1 komentar yang ada emoji
sample_comment = None
for i in range(len(df)):
    comment = df.iloc[i]['comment']
    if any(ord(char) > 127 for char in str(comment)):
        sample_comment = comment
        break

if sample_comment:
    print(f"KOMENTAR CONTOH (ADA EMOJI): '{sample_comment}'")
    
    # Test method ASCII encoding
    cleaned_ascii = sample_comment.encode('ascii', 'ignore').decode('ascii')
    print(f"Setelah ASCII encode/decode: '{cleaned_ascii}'")
    
    # Cek apakah masih ada non-ASCII
    still_has_emoji = any(ord(char) > 127 for char in cleaned_ascii)
    print(f"Masih ada emoji? {still_has_emoji}")
    
    if still_has_emoji:
        print("❌ MASIH ADA EMOJI SETELAH ASCII ENCODING!")
        # Tampilkan karakter yang masih ada
        problematic = [(char, ord(char)) for char in cleaned_ascii if ord(char) > 127]
        print(f"Karakter bandel: {problematic}")
else:
    print("✅ Tidak ditemukan komentar dengan emoji")

In [ ]:
# Cell 3 - Cleaning dengan Debug Print
print("🧼 CLEANING DENGAN DEBUG...")

def remove_noise_debug(text):
    if pd.isna(text):
        return ""
    
    original_text = str(text)
    print(f"   Processing: '{original_text}'")
    
    # Step 1: ASCII encoding
    text_ascii = original_text.encode('ascii', 'ignore').decode('ascii')
    print(f"   After ASCII: '{text_ascii}'")
    
    # Step 2: Remove URLs, mentions, hashtags
    text_clean = re.sub(r'http\S+|www\S+|https\S+', '', text_ascii)
    text_clean = re.sub(r'@\w+', '', text_clean)
    text_clean = re.sub(r'#\w+', '', text_clean)
    
    # Step 3: Remove punctuation and extra spaces
    text_clean = re.sub(r'[^\w\s]', ' ', text_clean)
    text_clean = re.sub(r'\s+', ' ', text_clean).strip()
    
    print(f"   Final: '{text_clean}'")
    print(f"   ---")
    
    return text_clean

# Test dengan 3 komentar pertama yang ada emoji
emoji_comments = []
for i in range(len(df)):
    comment = df.iloc[i]['comment']
    if any(ord(char) > 127 for char in str(comment)):
        emoji_comments.append(comment)
    if len(emoji_comments) >= 3:
        break

if emoji_comments:
    print("🔍 TEST DENGAN KOMENTAR BERTEMOI:")
    for comment in emoji_comments:
        remove_noise_debug(comment)
else:
    print("🔍 TEST DENGAN 3 KOMENTAR PERTAMA:")
    for i in range(min(3, len(df))):
        remove_noise_debug(df.iloc[i]['comment'])

# Apply ke semua data (tanpa debug print)
def remove_noise_final(text):
    if pd.isna(text):
        return ""
    
    text = str(text)
    text = text.encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

print("🎯 MENERAPKAN KE SEMUA DATA...")
df['comment_clean'] = df['comment'].apply(remove_noise_final)
print("✅ Cleaning selesai")

In [ ]:
# Cell 3.5 - Debug Emoji Bandel
print("🔍 DEBUG EMOJI BANDEL...")

# Cari semua komentar yang masih mengandung karakter non-ASCII
def has_non_ascii(text):
    if pd.isna(text):
        return False
    return any(ord(char) > 127 for char in str(text))

non_ascii_comments = df[df['comment_clean'].apply(has_non_ascii)]

if len(non_ascii_comments) > 0:
    print(f"❌ MASIH ADA {len(non_ascii_comments)} KOMENTAR DENGAN KARAKTER NON-ASCII:")
    
    for idx, row in non_ascii_comments[['comment', 'comment_clean']].head(10).iterrows():
        print(f"\n--- Komentar {idx} ---")
        print(f"ORIGINAL: {row['comment']}")
        print(f"CLEANED:  {row['comment_clean']}")
        
        # Tampilkan karakter problematic
        problematic = [(i, char, ord(char)) for i, char in enumerate(str(row['comment_clean'])) if ord(char) > 127]
        print(f"PROBLEMATIC CHARS: {problematic}")
        
else:
    print("✅ SELAMAT! Semua emoji dan karakter non-ASCII sudah hilang!")

In [ ]:
# Cell 4 - Normalisasi Teks (FIXED)
print("🔠 NORMALISASI TEKS...")

# Setup stemmer
stemmer = PorterStemmer()

# Stopwords manual untuk bahasa Indonesia + Inggris
stop_words = set([
    # Indonesian stopwords
    'yang', 'dan', 'di', 'dari', 'dengan', 'untuk', 'pada', 'ke', 'dalam',
    'ini', 'itu', 'saya', 'kamu', 'kita', 'mereka', 'dia', 'aku', 'engkau',
    'ada', 'adalah', 'akan', 'telah', 'sudah', 'lagi', 'juga', 'saja',
    'atau', 'tapi', 'namun', 'jika', 'karena', 'sehingga', 'oleh', 'agar',
    'supaya', 'meski', 'walau', 'sambil', 'seraya', 'setelah', 'sebelum',
    'ketika', 'sejak', 'sampai', 'hingga', 'apabila', 'asalkan', 'jikalau',
    'yaitu', 'yakni', 'bahwa', 'agar', 'supaya', 'sementara', 'selama',
    'hingga', 'bila', 'mana', 'sebab', 'oleh', 'karena', 'sehingga',
    'dll', 'dsb', 'dkk', 'dgn', 'yg', 'utk', 'pd', 'sdh', 'tdk', 'ga',
    
    # English stopwords  
    'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
    'with', 'by', 'is', 'are', 'was', 'were', 'be', 'been', 'have', 'has',
    'had', 'do', 'does', 'did', 'will', 'would', 'could', 'should', 'may',
    'might', 'must', 'can', 'shall', 'this', 'that', 'these', 'those',
    'you', 'your', 'yours', 'me', 'my', 'mine', 'he', 'him', 'his', 'she',
    'her', 'hers', 'it', 'its', 'we', 'us', 'our', 'ours', 'they', 'them',
    'their', 'theirs'
])

print(f"🔧 Menggunakan {len(stop_words)} stopwords")

def normalize_text(text):
    if pd.isna(text) or text == "":
        return ""
    
    text = str(text)
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. Hapus tanda baca dan angka
    text = re.sub(r'[^\w\s]', ' ', text)  # Hapus tanda baca
    text = re.sub(r'\d+', '', text)       # Hapus angka
    
    # 3. Tokenisasi (sederhana dengan split)
    tokens = text.split()
    
    # 4. Hapus stopwords dan kata terlalu pendek
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
    
    # 5. Stemming
    tokens = [stemmer.stem(word) for word in tokens]
    
    # 6. Join kembali
    cleaned_text = ' '.join(tokens)
    
    # Hapus extra spaces
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    
    return cleaned_text

# Test dulu dengan sample yang mengandung emoji
test_text = "Saya suka video ini! 😊 https://example.com @user #bagus"
result = normalize_text(test_text)
print(f"🔍 Test: '{test_text}'")
print(f"   -> '{result}'")

# Terapkan ke data yang SUDAH DIBERSIHKAN dari Cell 3
df['comment_normalized'] = df['comment_clean'].apply(normalize_text)
print("✅ Teks dinormalisasi")

# Tampilkan sample hasil
print("\n📋 SAMPLE HASIL NORMALISASI:")
sample_df = df[['comment', 'comment_clean', 'comment_normalized']].head(5)
for idx, row in sample_df.iterrows():
    print(f"   Original: {row['comment']}")
    print(f"   Cleaned:  {row['comment_clean']}")
    print(f"   Normalized: {row['comment_normalized']}")
    print("   ---")

In [ ]:
# Cell 5 - Simpan Hasil (FIXED)
print("💾 MENYIMPAN HASIL...")

# Hapus komentar kosong setelah normalisasi
sebelum = len(df)
df = df[df['comment_normalized'].str.len() > 0]
sesudah = len(df)
print(f"✅ Komentar kosong dihapus: {sebelum - sesudah}")

# Simpan data bersih
df.to_csv('comments_cleaned.csv', index=False)

print(f"✅ Data disimpan: {len(df)} komentar bersih")

# Tampilkan final sample
print("\n🎯 FINAL SAMPLE HASIL:")
final_samples = df[['comment', 'comment_normalized']].head(3)
for idx, row in final_samples.iterrows():
    print(f"   BEFORE: {row['comment']}")
    print(f"   AFTER:  {row['comment_normalized']}")
    print("   ---")

print(f"\n📊 TOTAL KOMENTAR BERSIH: {len(df)}")